[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ralbu85/cbm-colab-2026/blob/main/notebooks/2-1_%EC%84%BC%EC%84%9C%EC%8B%A0%ED%98%B8%EC%99%80_%ED%8A%B9%EC%A7%95%EC%B6%94%EC%B6%9C.ipynb)

# 2-1 · 유압 장치 데이터 파악하기, 그리고 좋은 특징 고르기

**⏱ 80분** · 모듈 2: 센서 → **특징** → 이상탐지 → 고장진단 → 판정 기준

## 🎯 이번 시간의 질문

1. 이 데이터는 **무엇을, 어떻게** 기록한 것인가?
2. 부품 상태를 알려면 센서의 **무엇을** 봐야 하는가?

1-1에서는 분석 전에 "한 행 = 디스크 1대의 하루", "90일을 다 채우지 못한 디스크가 5,407대"부터 확인했습니다. 데이터가 어떻게 만들어졌는지 모르고 분석하면 숫자를 잘못 읽습니다.
유압 데이터는 디스크 기록보다 훨씬 복잡합니다. 센서가 17개이고, 60초마다 수천 개 값이 쌓이며, 연구자가 순서를 정해 부품 상태를 바꿨습니다. 그래서 이번 시간의 **앞 절반은 데이터 파악**, **뒤 절반은 특징과 구분력**입니다.

**이번 시간에 익힐 패턴:** P1 `shape`·`head`·`describe` · `value_counts`·`crosstab`로 구성 파악 · 배열 요약(`mean`, `axis=1`) · P3 `groupby` · P4 상자그림 · 구분력 계산

## 0. 장치와 실험 (읽기, 5분)

### 장치 — 공장 유압 설비의 축소판

독일 ZeMA 연구소의 유압 시험장치입니다. 공장의 유압 프레스·사출기와 같은 구성으로, **주 회로**와 **냉각·여과 회로**가 기름 탱크로 연결되어 있습니다(아래는 단순화한 개념도).

```
[주 회로]         전동기 ──▶ 주 펌프 ──▶ 밸브 ──▶ 부하(실린더)      ← 일을 하는 회로
                              │(내부 누설)   │(전환)
                              └──── 축압기 (압력 변동을 받아 줌)
                                     │
                                 기름 탱크
                                     │
[냉각·여과 회로]  여과 펌프 ──▶ 필터 ──▶ 냉각기 ──▶ 탱크로          ← 기름을 식히고 거르는 회로
```

### 실험 — 60초 작업을 반복하며 부품을 일부러 나쁘게

- 장치는 **60초짜리 같은 작업 사이클**(정해진 순서로 부하를 올렸다 내림)을 계속 반복합니다.
- 연구자들이 네 부품의 상태를 **일부러 나쁜 쪽으로 조정**해 가며 사이클마다 센서를 기록했습니다. 사이클마다 "그때 네 부품의 상태"가 정답(라벨)으로 붙어 있습니다.

| 부품 | 열 이름 | 상태 값 (정상 → 고장 직전) | 나빠지면 |
|---|---|---|---|
| 냉각기 | `cooler` | 100 · 20 · 3 (%, 냉각 효율) | 기름이 과열 |
| 밸브 | `valve` | 100 · 90 · 80 · 73 (%, 전환 성능) | 유로 전환이 늦어짐 |
| 펌프 | `pump_leakage` | 0 · 1 · 2 (없음 · 약한 누설 · 심한 누설) | 내보내는 기름이 줄어듦 |
| 축압기 | `accumulator` | 130 · 115 · 100 · 90 (bar, 충전 압력) | 압력 변동을 못 받아 줌 |

### 센서 — 17개, 측정 속도가 서로 다름

| 센서 | 측정하는 것 | 단위 | 1초에 | 사이클(60초)당 값 |
|---|---|---|---|---|
| PS1 ~ PS6 | 압력 (6곳) | bar | 100번 | 6,000개 |
| EPS1 | 전동기 전력 | W | 100번 | 6,000개 |
| FS1, FS2 | 유량 (2곳) | L/min | 10번 | 600개 |
| TS1 ~ TS4 | 기름 온도 (4곳) | °C | 1번 | 60개 |
| VS1 | 진동 | mm/s | 1번 | 60개 |

(원자료에는 계산으로 만든 가상 센서 3개도 있지만, 실제 측정값이 아니라서 이 수업에서는 쓰지 않습니다.)

In [ ]:
# ▶ 준비 셀 — 그대로 실행하세요. (실습 데이터와 한글 글꼴을 내려받습니다)
import os, urllib.request, warnings
REPO = "https://raw.githubusercontent.com/ralbu85/cbm-colab-2026/main/"
for path in ["data/hydraulic_labels.csv",
             "data/hydraulic_features.csv",
             "data/hydraulic_raw_4sensors.npz",
             "fonts/NanumGothic.ttf"]:
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(REPO + path, path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_manager.fontManager.addfont("fonts/NanumGothic.ttf")
plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False,
                     "figure.figsize": (8, 4), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore")
print("준비 완료!")

## 1. 파일 세 개와 한 행의 의미

| 파일 | 내용 | 크기 |
|---|---|---|
| `hydraulic_labels.csv` | 사이클마다 네 부품의 상태(정답) | 2,205행 |
| `hydraulic_features.csv` | 사이클마다 17개 센서를 요약한 특징 70개 (5절에서 만드는 방법을 봄) | 2,205행 |
| `hydraulic_raw_4sensors.npz` | 센서 4개(PS1, FS1, TS1, VS1)의 원신호 전체 | 2,205 사이클 × 수천 개 |

In [ ]:
# ▶ 그대로 실행하세요
labels = pd.read_csv('data/hydraulic_labels.csv')
features = pd.read_csv('data/hydraulic_features.csv')
raw = np.load('data/hydraulic_raw_4sensors.npz')
PS1, FS1, TS1, VS1 = raw['PS1'], raw['FS1'], raw['TS1'], raw['VS1']
print('읽기 완료')

**P1 · 읽고 보기** — 정답 표부터 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(labels.shape)
labels.head()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — **한 행 = 장치 한 대의 60초 사이클 한 번**입니다. 2,205행은 장치 2,205대가 아니라 **한 대를 2,205번** 기록한 것입니다(1-1의 "523만 행 ≠ 523만 대"와 같은 주의).

| 열 | 뜻 |
|---|---|
| `cycle_id` | 사이클 번호 = **시간 순서** (1번이 가장 먼저) |
| `cooler` ~ `accumulator` | 그 사이클의 네 부품 상태 (정답) |
| `stable_flag` | 0 = 안정된 상태, 1 = 조건을 바꾼 직후라 아직 안정되지 않았을 수 있음 |
| `condition_group` | 네 부품 상태의 조합 이름 (예: `3_100_0_130` = 냉각기 3, 밸브 100, 누설 0, 축압기 130) |
| `split` | 학습용(train) / 평가용(test) 구분 (2절 끝에서 설명) |

특징 표는 같은 `cycle_id`로 한 행씩 짝이 맞습니다. 두 표를 합쳐 `data`로 씁니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(features.shape)
data = labels.merge(features, on='cycle_id')
print(data.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 특징 표는 `cycle_id` + 특징 70개 = 71열이고, 합친 `data`는 2,205행 × 78열입니다.

## 2. 실험 설계 파악 — 어떤 상태를 몇 번씩, 어떤 순서로

**부품 상태별 사이클 수** — `value_counts()`를 네 부품에 반복합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
for part in ['cooler', 'valve', 'pump_leakage', 'accumulator']:
    print(data[part].value_counts().sort_index())
    print()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설**
- 냉각기는 세 상태가 약 730사이클씩 고르게 있습니다. 밸브는 정상(100)이 1,125, 펌프는 누설 없음이 1,221로 가장 많고, 축압기는 고장 직전(90)이 808로 가장 많습니다.
- 부품마다 **정상이 아닌 상태의 사이클이 45~73%**나 됩니다(냉각기 66%, 밸브 49%, 펌프 45%, 축압기 73%). 시험장치라서 가능한 일입니다. 현장에서는 고장이 드물어서(1-1: 디스크 0.33%) 이렇게 모으기 어렵습니다. 이 차이가 2-2(정상 데이터만으로 경보 만들기)의 출발점입니다.

**부품 상태를 어떻게 섞었나** — 두 부품의 교차표와, 네 부품 조합의 가짓수를 셉니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(pd.crosstab(data['cooler'], data['pump_leakage']))
print('상태 조합 수:', data['condition_group'].nunique())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기 × 펌프 누설의 **모든 칸에 사이클이 있고**, 네 부품의 조합은 3 × 4 × 3 × 4 = **144가지 전부**입니다. 부품 상태를 서로 **독립적으로 섞어서** 시험했다는 뜻입니다.
그래서 "냉각기가 정상인 사이클"에도 펌프 누설·밸브 지연·축압기 저압이 섞여 있습니다. **한 부품을 볼 때 다른 부품의 영향이 함께 들어온다**는 점이 이 데이터를 읽는 핵심입니다(2-2, 2-3에서 계속 등장).

**실험 순서** — 네 부품의 상태를 사이클 번호(시간) 순서로 그린 "실험 일정표"입니다. (▶ 위는 전체, 아래는 200~500번 확대)

In [ ]:
# ▶ 그대로 실행하세요
parts = [('cooler', '냉각기 (%)'), ('valve', '밸브 (%)'), ('pump_leakage', '펌프 누설'), ('accumulator', '축압기 (bar)')]
for x_range, title in [((1, 2205), '실험 일정표 — 전체 2,205 사이클'), ((200, 500), '확대 — 200~500번 사이클')]:
    fig, axes = plt.subplots(4, 1, figsize=(11, 6), sharex=True)
    for ax, (part, name) in zip(axes, parts):
        ax.step(data['cycle_id'], data[part], where='post')
        ax.set_ylabel(name, fontsize=9)
    axes[-1].set_xlim(*x_range)
    axes[-1].set_xlabel('사이클 번호 (시간 순서)')
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

💬 **결과 해설** — 부품마다 바꾼 속도가 다릅니다.
- **냉각기**: 3% → 20% → 100%의 **큰 세 덩어리**(약 730사이클씩)로 천천히 바꿨습니다.
- **축압기**: 130 → 115 → 100 → 90을 약 730사이클마다 한 바퀴씩, 모두 세 바퀴 돌았습니다(냉각기가 바뀌는 시점과는 조금 어긋남).
- **펌프 누설**은 약 40~50사이클마다, **밸브**는 약 10사이클마다 바뀝니다(확대 그림).
- 각 바퀴의 처음 약 200사이클은 밸브·펌프를 정상으로 두고 **길게** 돌렸습니다.

이 일정표에서 두 가지를 알 수 있습니다.
1. **같은 조합이 연달아 기록**되어 이웃한 사이클은 거의 쌍둥이입니다. 학습용과 평가용을 무작위로 나누면 쌍둥이가 양쪽에 들어가 **시험 문제를 미리 본 셈**이 됩니다. 그래서 이 수업은 **같은 상태 조합은 한쪽에만** 들어가도록 나눠 두었습니다(`split`).
2. 상태를 **바꾼 직후**에는 기름 온도 등이 새 상태로 옮겨 가는 중이라 센서 값이 어중간합니다. 이런 사이클이 `stable_flag = 1`입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print(data['split'].value_counts())
print(data['stable_flag'].value_counts())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 학습용 1,869 / 평가용 336 사이클입니다. 평가용 336개는 모델이 **한 번도 보지 못한 상태 조합**입니다. 전체의 약 1/3(756사이클)은 조건을 바꾼 직후의 과도 구간입니다.

## 3. 센서 값 파악 — 단위, 범위, 결측

센서 다섯 개의 60초 평균이 어떤 범위에 있는지 `describe()`로 봅니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data[['PS1_mean', 'EPS1_mean', 'FS1_mean', 'TS1_mean', 'VS1_mean']].describe().round(2)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설**
- 압력 PS1은 약 155~181 bar, 전동기 전력 EPS1은 약 2,360~2,740 W, 유량 FS1은 2.0~6.7 L/min, 기름 온도 TS1은 35~58°C, 진동 VS1은 0.5~0.8 mm/s입니다.
- 센서마다 **단위와 크기가 수천 배 다릅니다.** 전력의 "100 차이"와 진동의 "0.1 차이" 중 어느 쪽이 큰 변화인지 숫자만으로는 알 수 없습니다. 그래서 뒤에서는 값을 그대로 비교하지 않고 **흩어짐(표준편차)으로 나눈 값**(구분력, z-점수)으로 비교합니다.
- 유량 FS1의 최솟값 2.0은 평균 6.2에서 멀리 떨어져 있습니다. 어떤 상태에서 유량이 크게 줄어드는지는 뒤에서 확인합니다.

**결측 확인** — 1-1에서는 SMART 값 -1이 "보고 안 됨"이었습니다. 이 데이터에는 빈칸이 있을까요?

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.isna().sum().sum()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 0. 빈칸은 없습니다. 하지만 빈칸이 없다고 **값이 모두 믿을 만하다는 뜻은 아닙니다**(과제 1에서 확인).

## 4. 원신호 한 사이클 보기

원신호는 **2차원 배열**입니다. 행 = 사이클, 열 = 사이클 안의 시간 순서입니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
print('PS1:', PS1.shape)
print('TS1:', TS1.shape)
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 압력 PS1은 (2205, 6000): 1초에 100번 × 60초 = **사이클마다 6,000개**. 온도 TS1은 1초에 1번이라 60개입니다.

**P4 · 선그래프** — 첫 사이클(`PS1[0]`)의 압력을 시간 축으로 그립니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
t = np.arange(6000) / 100
plt.plot(t, PS1[0])
plt.xlabel('사이클 안의 시간 (초)')
plt.ylabel('압력 PS1 (bar)')
plt.title('사이클 1의 압력 신호')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 압력이 계단처럼 오르내립니다. 장치가 60초 동안 정해진 순서로 부하를 바꾸기 때문이고, 모든 사이클이 같은 순서를 반복합니다. 네 센서를 함께 봅니다.

In [ ]:
# ▶ 그대로 실행하세요
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, sig, hz, unit) in zip(axes.flat, [('PS1 압력', PS1, 100, 'bar'), ('FS1 유량', FS1, 10, 'L/min'),
                                                   ('TS1 온도', TS1, 1, '°C'), ('VS1 진동', VS1, 1, 'mm/s')]):
    ax.plot(np.arange(sig.shape[1]) / hz, sig[0])
    ax.set_title(name); ax.set_xlabel('시간 (초)'); ax.set_ylabel(unit)
fig.tight_layout()
plt.show()

💬 유량은 압력처럼 계단 모양, 온도는 한 사이클 안에서 거의 일정, 진동은 작게 흔들립니다. 이 긴 신호를 사이클끼리 그대로 비교하기는 어렵습니다.

## 5. 특징 만들기 — 6,000개 값을 숫자 몇 개로

| 특징 | 의미 |
|---|---|
| 평균 `mean()` | 신호의 전체 수준 |
| 표준편차 `std()` | 신호가 흔들리는 크기 |
| 최댓값 `max()`, 최솟값 `min()` | 극단값 |

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
x = PS1[0]
print('평균:', x.mean())
print('표준편차:', x.std())
print('최댓값:', x.max())
print('최솟값:', x.min())
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 사이클 1의 압력 6,000개가 네 숫자(평균 160.7, 표준편차 13.9, 최대 191.5, 최소 145.8)로 줄었습니다.

**모든 사이클을 한 번에** — `TS1.mean(axis=1)`은 **각 행(사이클)마다 가로 방향으로** 평균을 내서 2,205개의 값을 돌려줍니다. 제공된 `TS1_mean` 열과 같은지 확인합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
my_ts1 = TS1.mean(axis=1)
print(my_ts1[:3])
print(data['TS1_mean'].values[:3])
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 두 줄의 숫자가 같으면 됩니다(35.62, 36.68, 37.88).

💬 제공된 `features` 표는 이 방법으로 **17개 센서 × 5개 특징**(평균 `mean`, 표준편차 `std`, 최솟값 `min`, 최댓값 `max`, 마지막 1초 − 첫 1초 `change`) = 70개 열을 미리 계산해 둔 것입니다.

## 6. 좋은 특징이란 — 상자그림과 구분력

**P3 · 묶어 요약** — 냉각기 상태별 오일 온도 평균.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.groupby('cooler')['TS1_mean'].mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


**P4 · 상자그림** — 상태별로 값의 범위(상자)와 가운데(선)를 보여 줍니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
data.boxplot(column='TS1_mean', by='cooler')
plt.suptitle('')          # pandas가 자동으로 붙이는 윗제목 지우기
plt.xlabel('냉각기 상태 (%)')
plt.ylabel('오일 온도 TS1 평균 (°C)')
plt.show()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기가 나빠질수록(100 → 20 → 3) 오일 온도가 약 **36 → 45 → 55°C**로 오르고, 세 상자가 겹치지 않습니다. 냉각이 안 되면 기름이 뜨거워지는 물리 관계 그대로입니다.
상자 밖의 동그라미는 냉각기 상태를 **바꾼 직후** 기름 온도가 아직 옮겨 가는 중인 사이클입니다(현장의 정비 직후·조건 변경 직후 과도 구간과 같음).

### 구분력 — 특징이 얼마나 좋은지를 숫자 하나로

상자그림을 70개 특징 × 4개 부품마다 그려 볼 수는 없습니다. 그래서 "**상자 사이의 거리가 상자 크기의 몇 배인가**"를 숫자로 계산합니다.

$$\text{구분력} = \frac{\text{상태별 평균 중 가장 큰 값} - \text{가장 작은 값}}{\text{상태 안 표준편차의 평균}}$$

| 구분력 | 의미 |
|---|---|
| 3 이상 | 상자가 거의 겹치지 않음 → 특징 하나로 상태를 거의 알 수 있음 |
| 1 근처 | 상자가 많이 겹침 → 특징 하나로는 부족 |

한 번은 손으로 계산합니다.

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
g = data.groupby('cooler')['TS1_mean']
print(g.mean())
print(g.std())
(g.mean().max() - g.mean().min()) / g.std().mean()
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


✅ **체크포인트** — 구분력 약 **11.4**.

💬 가장 먼 두 상태의 온도 차이(55.0 − 36.0 = 19.0°C)가 상태 안 흩어짐(평균 1.7°C)의 11배입니다. 상자가 겹치지 않을 수밖에 없습니다.

### ▶ 구분력 함수

같은 계산을 모든 특징에 반복하도록 함수로 묶었습니다.
- `separation(부품, 특징)` → 구분력 하나
- `rank_features(부품)` → 70개 특징의 구분력을 계산해 **높은 순서로 5개**

In [ ]:
# ▶ 그대로 실행하세요
def separation(part, feature):
    """구분력 = (상태별 평균의 최댓값 − 최솟값) ÷ (상태 안 표준편차의 평균)."""
    g = data.groupby(part)[feature]
    return round((g.mean().max() - g.mean().min()) / g.std().mean(), 2)

def rank_features(part, top=5):
    """70개 센서 특징의 구분력을 높은 순서로 보여 준다."""
    cols = [c for c in features.columns if c != 'cycle_id']
    return pd.Series({c: separation(part, c) for c in cols}).sort_values(ascending=False).head(top)

**✍️ 직접 입력** — 아래 코드를 보고 **직접 타이핑**해서 실행하세요. (복사·붙여넣기 말고 손으로!)

```python
rank_features('cooler')
```

In [ ]:
# ✍️ 위 코드를 여기에 직접 입력하세요


💬 **결과 해설** — 냉각기에는 구분력 13 안팎의 특징이 여럿 있습니다. 온도(TS4)뿐 아니라 압력 PS5·PS6도 높은데, 기름이 뜨거워지면 묽어져 압력에도 영향을 주기 때문으로 보입니다.
**냉각기는 "좋은 특징"이 많은 부품**입니다. 다른 부품도 그럴까요? → 과제 2.

---
## 7. 실습 과제 (30분)

### 📝 과제 1 · 압력 PS4의 0은 무엇일까? (데이터 파악)
압력 센서 PS4의 60초 평균(`PS4_mean`)을 보면 0인 사이클이 많습니다. **몇 개**인지 세고, **냉각기 상태와 관계가 있는지** 교차표로 확인하세요. 그리고 이 0을 어떻게 다뤄야 할지 적으세요.

| 냉각기 상태 | PS4 = 0인 사이클 | 전체 사이클 |
|---|---|---|
| 3% | | 732 |
| 20% | | 732 |
| 100% | | 741 |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: (data['PS4_mean'] == 0).sum()  — 조건이 참인 개수
# 힌트 2: pd.crosstab(data['cooler'], data['PS4_mean'] == 0)


✏️ **나의 답:**

### 📝 과제 2 · 부품별로 가장 좋은 특징 찾기
`rank_features()`로 나머지 세 부품의 1위 특징과 구분력을 찾아 표를 채우세요. **실행 전에** 어느 부품이 가장 어려울지 예상해 보세요.

| 부품 | 1위 특징 | 구분력 | 특징 하나로 충분한가? |
|---|---|---|---|
| 냉각기 `cooler` | PS6_min | 13.23 | 예 |
| 펌프 `pump_leakage` | | | |
| 밸브 `valve` | | | |
| 축압기 `accumulator` | | | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: rank_features('pump_leakage') 처럼 부품 이름만 바꿉니다.


✏️ **나의 답:**

### 📝 과제 3 · 펌프 누설의 1위 특징을 눈으로 확인
과제 2에서 찾은 펌프의 1위 특징(모터 전력 `EPS1_mean`)을 누설 상태별 상자그림으로 그리세요. 냉각기-온도 상자그림과 무엇이 다른가요?

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: 6절의 상자그림 코드에서 'TS1_mean' → 'EPS1_mean', 'cooler' → 'pump_leakage'


✏️ **나의 답:**

### 📝 과제 4 · 밸브는 파형의 어디에서 다를까?
밸브는 1위 특징도 구분력 1.2로 약했습니다. 60초 전체를 요약한 특징이 놓친 것이 있을 수 있습니다.
아래 네 사이클은 **다른 부품은 모두 정상**이고 밸브만 다릅니다: 사이클 1788(밸브 100), 1778(90), 1768(80), 1758(73).
네 사이클의 PS1 압력을 겹쳐 그리고, **어느 시간대에서 차이가 나는지** 찾으세요. (배열 행 번호는 0부터: 사이클 1788 → `PS1[1787]`)

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트 1: plt.plot(t, PS1[1787], label='밸브 100') 을 네 번(label을 다르게) 부른 뒤 plt.legend()
# 힌트 2: 차이가 잘 안 보이면 plt.xlim(8, 12) 로 8~12초만 확대합니다.


✏️ **나의 답:**

### 📝 과제 5 · 새 특징을 만들어 구분력 비교 (도전)
과제 4에서 찾은 구간으로 **"9~10초 구간의 압력 평균"** 특징을 모든 사이클에 대해 만들고, 구분력을 기존 특징과 비교하세요.
9~10초는 PS1 배열의 900번째~999번째 열입니다: `PS1[:, 900:1000]` (`:` = 모든 행)

| 밸브 진단용 특징 | 구분력 |
|---|---|
| PS1_mean (60초 평균) | |
| FS1_max (기존 1위) | 1.18 |
| PS1_peak9 (새 특징) | |

In [ ]:
# 📝 과제 — 여기에 코드를 작성하세요
# 힌트: data['PS1_peak9'] = PS1[:, 900:1000].mean(axis=1)  →  separation('valve', 'PS1_peak9')


✏️ **나의 답:**

---
## ✅ 정리

| 배운 것 | 핵심 |
|---|---|
| 한 행의 의미 | 장치 1대의 60초 사이클 1번. 행 수 ≠ 설비 수 |
| 실험 설계 | 네 부품 상태를 독립적으로 섞은 144조합. 같은 조합이 연달아 → 조합 단위로 학습/평가 분리 · 상태 변경 직후는 과도 구간 |
| 센서 값 | 단위·크기가 수천 배 차이 → 흩어짐으로 나눠 비교. 빈칸이 없어도 값을 의심(PS4의 0) |
| 특징 추출 | 원신호 6,000개 → 평균·표준편차·최대·최소. `배열.mean(axis=1)`로 모든 사이클 한 번에 |
| 구분력 | 상자 사이 거리 ÷ 상자 크기. 3 이상이면 특징 하나로 충분, 1 근처면 부족 |
| 부품별 결과 | 냉각기 13 · 펌프 1.1 · 밸브 1.2 · 축압기 1.2. **냉각기만 쉬운 부품** |
| 특징 설계 | 전체 평균에 안 보이는 차이가 특정 구간에 숨어 있다 (밸브 9~10초 → 구분력 8.9) |

다음 질문: 구분력이 높은 냉각기는 **정상 데이터만으로도** 경보를 만들 수 있을까요? 구분력이 1밖에 안 되는 펌프 누설은요? → **2-2 · 이상탐지**